In [8]:
!pip install transformers datasets torch scikit-learn tqdm conllu

In [2]:
import torch
import numpy as np
from tqdm import tqdm

from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset

from sklearn.metrics.pairwise import cosine_similarity

In [16]:
model_name = "bert-base-multilingual-cased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)
model.eval()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(119547, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
 

In [9]:
from conllu import parse

parsed_datasets = {}
for lang in datasets.keys():
    with open(f"data/{lang}.conllu", "r", encoding="utf-8") as f:
        data = f.read()
    parsed_datasets[lang] = parse(data)

print("Datasets parsed successfully.")
# Example: Print the first sentence of the Hindi dataset
# print(parsed_datasets['hindi'][0].metadata['text'])


Datasets parsed successfully.


In [6]:
import os
import requests

datasets = {
"hindi":"https://raw.githubusercontent.com/UniversalDependencies/UD_Hindi-HDTB/master/hi_hdtb-ud-train.conllu",
"telugu":"https://raw.githubusercontent.com/UniversalDependencies/UD_Telugu-MTG/master/te_mtg-ud-train.conllu",
"tamil":"https://raw.githubusercontent.com/UniversalDependencies/UD_Tamil-TTB/master/ta_ttb-ud-train.conllu",
"marathi":"https://raw.githubusercontent.com/UniversalDependencies/UD_Marathi-UFAL/master/mr_ufal-ud-train.conllu"
}

os.makedirs("data",exist_ok=True)

for lang,url in datasets.items():

    r=requests.get(url)

    with open(f"data/{lang}.conllu","wb") as f:
        f.write(r.content)

print("Universal Dependencies datasets downloaded")

Universal Dependencies datasets downloaded


In [7]:
# The datasets are now downloaded locally. We will process them from local files if needed.

In [11]:
datasets = [
    ("hindi",parsed_datasets['hindi']),
    ("tamil",parsed_datasets['tamil']),
    ("telugu",parsed_datasets['telugu']),
    ("marathi",parsed_datasets['marathi'])
]

In [12]:
def get_embeddings(words):

    inputs = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    hidden = outputs.last_hidden_state.squeeze(0)

    word_ids = inputs.word_ids()

    embeddings = []
    current_tokens = []
    last_word = None

    for token_idx, word_idx in enumerate(word_ids):

        if word_idx is None:
            continue

        if last_word is None:
            last_word = word_idx

        if word_idx != last_word:

            embeddings.append(np.mean(current_tokens,axis=0))

            current_tokens = []
            last_word = word_idx

        current_tokens.append(hidden[token_idx].cpu().numpy())

    if current_tokens:
        embeddings.append(np.mean(current_tokens,axis=0))

    return embeddings

In [13]:
pos_embeddings = []
pos_labels = []
pos_languages = []

MAX_SENTENCES = 2000

In [17]:
for lang,dataset in datasets:

    print("Processing",lang)

    for i in tqdm(range(min(len(dataset),MAX_SENTENCES))):

        sentence_tokens = dataset[i] # dataset[i] is a TokenList (list of token dictionaries)
        words = [token['form'] for token in sentence_tokens]
        pos = [token['upos'] for token in sentence_tokens]

        emb = get_embeddings(words)

        for e,p in zip(emb,pos):

            pos_embeddings.append(e)
            pos_labels.append(p)
            pos_languages.append(lang)

Processing hindi


100%|██████████| 2000/2000 [00:26<00:00, 76.53it/s]


Processing tamil


100%|██████████| 400/400 [00:08<00:00, 48.36it/s]


Processing telugu


100%|██████████| 1051/1051 [00:10<00:00, 98.22it/s] 


Processing marathi


100%|██████████| 373/373 [00:03<00:00, 99.69it/s]


In [18]:
pos_embeddings = np.array(pos_embeddings)

pos_embeddings = pos_embeddings / np.linalg.norm(
    pos_embeddings,
    axis=1,
    keepdims=True
)

In [19]:
def predict_pos(sentence, k=7):

    emb = get_embeddings(sentence)

    results = []

    for e in emb:

        e = e / np.linalg.norm(e)

        sims = cosine_similarity(
            [e],
            pos_embeddings
        )[0]

        top_k = sims.argsort()[-k:]

        tags = []
        langs = []

        for idx in top_k:

            tags.append(pos_labels[idx])
            langs.append(pos_languages[idx])

        pred = max(set(tags), key=tags.count)

        confidence = tags.count(pred)/k

        results.append((pred,confidence))

    return list(zip(sentence,results))

In [20]:
def show_prediction(sentence):

    preds = predict_pos(sentence)

    print("\nSentence:", " ".join(sentence))
    print("-----------------------------")

    for word,(pos,conf) in preds:

        print(f"{word:10} → {pos:6} confidence={conf:.2f}")

In [21]:
show_prediction(["nang","mara","tin"])


Sentence: nang mara tin
-----------------------------
nang       → PRON   confidence=0.43
mara       → PRON   confidence=0.57
tin        → NOUN   confidence=0.43
